# 01 Dataset Preparation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`02_official_split_dataset.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/02_official_split_dataset.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)
- [`convert_hand_made.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/convert_hand_made.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)
- [`rebuild_ml04_v3.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/rebuild_ml04_v3.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'02_data/scripts/02_official_split_dataset.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '06607f9de382a8cc70176f4d84beffe4752e6bcd'}, '02_data/scripts/convert_hand_made.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '97d6e17ebabec7248b1627eb17567ef0817eb205'}, '02_data/scripts/rebuild_ml04_v3.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '15aec4a103898860629814ab87e6b10c9572aff8'}}

## 02_official_split_dataset

원본: [02_data/scripts/02_official_split_dataset.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/02_official_split_dataset.ipynb)  
Git blob: `06607f9de382a8cc70176f4d84beffe4752e6bcd`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 실행은 아래 고정 커밋 원본을 사용한다.

### 원본 셀 0

# MNIST + EMNIST 공식 분할 기반 16클래스 데이터셋 재구성

- MNIST 공식 Training 60,000 / Test 10,000 유지
- EMNIST 대문자 A~F 각각 Training 6,000 / Test 1,000
- EMNIST Test: 공식 Test 우선, 부족하면 미사용 공식 Training 원본으로 보충
- EMNIST Training 부족분: Training에 남은 원본으로만 이동 증강
- 원본·증강과 출처를 기록하며 Test 누수 방지

**주의:** 교수님 프로젝트 가이드의 초기 학습용 Training/Validation 기준과 팀이 요청한 Training/Test 기준의 차이는 팀 내 확인이 필요합니다.

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell1
import numpy as np
from torchvision.datasets import MNIST, EMNIST
from scipy.ndimage import shift

SEED = 42
rng = np.random.default_rng(SEED)

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
emnist_train = EMNIST(root="./data", split="byclass", train=True, download=True)
emnist_test = EMNIST(root="./data", split="byclass", train=False, download=True)

print("원본 데이터셋 준비 완료")

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell2
print("MNIST 공식 구분")
for i in range(10):
    tr = int((mnist_train.targets == i).sum())
    te = int((mnist_test.targets == i).sum())
    print(f"{i}: Train {tr}, Test {te}")

print("\nEMNIST A~F 공식 구분")
for i, letter in enumerate("ABCDEF", start=10):
    tr = int((emnist_train.targets == i).sum())
    te = int((emnist_test.targets == i).sum())
    print(f"{letter}: Train {tr}, Test {te}, Test 부족 {max(0, 1000 - te)}")

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell3
X_train_parts = [mnist_train.data.numpy().astype(np.uint8)]
y_train_parts = [mnist_train.targets.numpy().astype(np.uint8)]
X_test_parts = [mnist_test.data.numpy().astype(np.uint8)]
y_test_parts = [mnist_test.targets.numpy().astype(np.uint8)]
train_is_augmented_parts = [np.zeros(len(mnist_train), dtype=np.bool_)]
train_source_split_parts = [np.zeros(len(mnist_train), dtype=np.uint8)]
test_source_split_parts = [np.ones(len(mnist_test), dtype=np.uint8)]
train_source_index_parts = [np.arange(len(mnist_train), dtype=np.int32)]
test_source_index_parts = [np.arange(len(mnist_test), dtype=np.int32)]

emnist_log = []
for label, letter in enumerate("ABCDEF", start=10):
    train_indices = np.flatnonzero(emnist_train.targets.numpy() == label)
    test_indices = np.flatnonzero(emnist_test.targets.numpy() == label)
    n_official_test = min(1000, len(test_indices))
    picked_test = rng.choice(test_indices, size=n_official_test, replace=False)
    n_borrowed = 1000 - n_official_test
    borrowed = rng.choice(train_indices, size=n_borrowed, replace=False)
    available_train = np.setdiff1d(train_indices, borrowed, assume_unique=False)
    n_original_train = min(6000, len(available_train))
    picked_train = rng.choice(available_train, size=n_original_train, replace=False)
    n_aug = 6000 - n_original_train

    train_images = emnist_train.data[picked_train].numpy().transpose(0, 2, 1).copy()
    test_images_official = emnist_test.data[picked_test].numpy().transpose(0, 2, 1).copy()
    test_images_borrowed = emnist_train.data[borrowed].numpy().transpose(0, 2, 1).copy()
    test_images = np.concatenate([test_images_official, test_images_borrowed])

    augmented_images = []
    augmented_parent = []
    if n_aug:
        parent_positions = rng.choice(n_original_train, size=n_aug, replace=True)
        for pos in parent_positions:
            dx = int(rng.choice([-2, -1, 1, 2]))
            dy = int(rng.choice([-2, -1, 1, 2]))
            moved = shift(train_images[pos], shift=(dy, dx), order=0, mode="constant", cval=0)
            augmented_images.append(moved.astype(np.uint8))
            augmented_parent.append(int(picked_train[pos]))
        augmented_images = np.stack(augmented_images)
        train_images_final = np.concatenate([train_images, augmented_images])
    else:
        train_images_final = train_images

    X_train_parts.append(train_images_final)
    y_train_parts.append(np.full(6000, label, dtype=np.uint8))
    train_is_augmented_parts.append(np.r_[np.zeros(n_original_train, dtype=np.bool_), np.ones(n_aug, dtype=np.bool_)])
    train_source_split_parts.append(np.zeros(6000, dtype=np.uint8))
    train_source_index_parts.append(np.r_[picked_train, np.array(augmented_parent, dtype=np.int32)])
    X_test_parts.append(test_images)
    y_test_parts.append(np.full(1000, label, dtype=np.uint8))
    test_source_split_parts.append(np.r_[np.ones(n_official_test, dtype=np.uint8), np.zeros(n_borrowed, dtype=np.uint8)])
    test_source_index_parts.append(np.r_[picked_test, borrowed].astype(np.int32))
    assert len(train_images_final) == 6000 and len(test_images) == 1000
    assert not np.intersect1d(np.r_[picked_train, np.asarray(augmented_parent, dtype=np.int32)], borrowed).size
    emnist_log.append((letter, len(train_indices), len(test_indices), n_original_train, n_aug, n_official_test, n_borrowed))

X_train_final = np.concatenate(X_train_parts)
y_train_final = np.concatenate(y_train_parts)
X_test_final = np.concatenate(X_test_parts)
y_test_final = np.concatenate(y_test_parts)
train_is_augmented = np.concatenate(train_is_augmented_parts)
train_source_split = np.concatenate(train_source_split_parts)
test_source_split = np.concatenate(test_source_split_parts)
train_source_index = np.concatenate(train_source_index_parts)
test_source_index = np.concatenate(test_source_index_parts)

print("문자 | 공식 Train | 공식 Test | Train 원본 | Train 증강 | Test 공식 | Test Train차용")
for item in emnist_log:
    print("{:>2} | {:>10} | {:>9} | {:>10} | {:>10} | {:>9} | {:>13}".format(*item))

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell4
assert X_train_final.shape == (96000, 28, 28)
assert X_test_final.shape == (16000, 28, 28)
assert np.array_equal(np.bincount(y_train_final, minlength=16)[:10], np.bincount(mnist_train.targets.numpy(), minlength=10))
assert np.array_equal(np.bincount(y_test_final, minlength=16)[:10], np.bincount(mnist_test.targets.numpy(), minlength=10))
assert np.all(np.bincount(y_train_final, minlength=16)[10:] == 6000)
assert np.all(np.bincount(y_test_final, minlength=16)[10:] == 1000)
assert X_train_final.dtype == np.uint8 and X_test_final.dtype == np.uint8

for label in range(16):
    train_ids = set(zip(train_source_split[y_train_final == label].tolist(), train_source_index[y_train_final == label].tolist()))
    test_ids = set(zip(test_source_split[y_test_final == label].tolist(), test_source_index[y_test_final == label].tolist()))
    assert train_ids.isdisjoint(test_ids), f"원본 누수: label={label}"

assert not np.any(train_is_augmented[0:60000])
assert int(train_is_augmented.sum()) == sum(row[4] for row in emnist_log)
print("Training:", X_train_final.shape, "Test:", X_test_final.shape)
print("증강 이미지 수:", int(train_is_augmented.sum()))
print("Test 중 공식 Training에서 차용한 EMNIST 원본 수:", sum(row[6] for row in emnist_log))
print("공식 분할 유지 및 데이터 누수 검증 완료")

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell5
filename = "MNIST_EMNIST_official_split_112000.npz"
np.savez_compressed(
    filename,
    X_train=X_train_final, y_train=y_train_final,
    X_test=X_test_final, y_test=y_test_final,
    train_is_augmented=train_is_augmented,
    train_source_split=train_source_split,
    test_source_split=test_source_split,
    train_source_index=train_source_index,
    test_source_index=test_source_index,
    emnist_class_log=np.array(emnist_log, dtype="U16"),
    seed=np.array(SEED),
)
with np.load(filename) as check:
    assert check["X_train"].shape == (96000, 28, 28)
    assert check["X_test"].shape == (16000, 28, 28)
print("파일 저장 및 재검증 완료:", filename)

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell6
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, letter in enumerate("ABCDEF"):
    label = i + 10
    tr = np.flatnonzero((y_train_final == label) & (~train_is_augmented))[0]
    te = np.flatnonzero(y_test_final == label)[0]
    axes[0, i].imshow(X_train_final[tr], cmap="gray")
    axes[0, i].set_title(f"Train {letter}")
    axes[1, i].imshow(X_test_final[te], cmap="gray")
    axes[1, i].set_title(f"Test {letter}")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell7
import numpy as np

with np.load("MNIST_EMNIST_official_split_112000.npz") as data:
    print("저장된 항목:", data.files)

    for key in data.files:
        print(key, data[key].shape)

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell8
import numpy as np

with np.load("MNIST_EMNIST_official_split_112000.npz") as data:
    train_labels = data["y_train"]
    test_labels = data["y_test"]
    train_splits = data["train_source_split"]
    test_splits = data["test_source_split"]
    train_indices = data["train_source_index"]
    test_indices = data["test_source_index"]

    train_sources = set(zip(
        train_labels.tolist(),
        train_splits.tolist(),
        train_indices.tolist()
    ))

    test_sources = set(zip(
        test_labels.tolist(),
        test_splits.tolist(),
        test_indices.tolist()
    ))

    overlap = train_sources & test_sources

    print("Training/Test 원본 중복:", len(overlap), "개")

    for i, letter in enumerate("ABCDEF"):
        label = i + 10
        print(
            f"{letter}:",
            "Training", np.sum(train_labels == label),
            "/ Test", np.sum(test_labels == label)
        )

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell9
import numpy as np

with np.load("MNIST_EMNIST_official_split_112000.npz") as data:
    labels = data["y_test"]
    splits = data["test_source_split"]
    augmented = data["train_is_augmented"]
    train_labels = data["y_train"]

    print("EMNIST 클래스별 Test 구성")

    for i, letter in enumerate("ABCDEF"):
        label = i + 10
        mask = labels == label

        values, counts = np.unique(splits[mask], return_counts=True)
        source_counts = dict(zip(values.tolist(), counts.tolist()))

        print(
            f"{letter}: Test 출처 {source_counts}, "
            f"Training 증강 {np.sum(augmented & (train_labels == label))}개"
        )

In [ ]:
%%archive_source 02_data/scripts/02_official_split_dataset.ipynb#cell10


## convert_hand_made

원본: [02_data/scripts/convert_hand_made.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/convert_hand_made.py)  
Git blob: `97d6e17ebabec7248b1627eb17567ef0817eb205`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)

In [ ]:
%%archive_source 02_data/scripts/convert_hand_made.py
#!/usr/bin/env python3
"""Recreate the current four standard Hand-made Grayscale NPZ files.

Requires BOTH the original bundle (writers 41/43/44) AND the later corrected
writer-42 submission ZIP. The original ML-Project.zip contains the superseded
writer-42 data; it must not silently overwrite the replacement.

Examples:
  python 02_data/scripts/convert_hand_made.py \\
    --source /path/to/ML-Project.zip \\
    --writer42-replacement '/path/to/ml26_42(2).zip' \\
    --output-dir 02_data/Hand_made
  python 02_data/scripts/convert_hand_made.py \\
    --input-dir /path/to/original_individual_zips \\
    --writer42-replacement '/path/to/ml26_42(2).zip' \\
    --output-dir 02_data/Hand_made

Current transformations:
- 41: CCW 90-degree rotation, no inversion
- 42 NEW: no rotation, no inversion; preserve the original submitted NPZ X
- 43: no rotation, no inversion
- 44: inversion (255-X) then CCW 90-degree rotation

Other than these documented transforms, no thresholding, smoothing, resize,
augmentation, or normalization is performed. Historical EXP-유-004 scores were
computed with the previous (superseded) 42 and must be reproduced separately
from its historical Git commit, not from this current dataset.
"""
from __future__ import annotations

import argparse
from collections import Counter
from hashlib import sha256
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
from PIL import Image

LABELS = '0123456789ABCDEF'
WRITERS = (41, 42, 43, 44)
ORIGINAL_WRITERS = (41, 43, 44)
INVERT_IDS = {44}
ROTATE_IDS = {41, 44}


def expected_files(writer: int) -> list[str]:
    return [f'{ch}_{i:02d}_{writer}.jpg' for i in range(1, 41) for ch in LABELS]


def read_original_writer(archive: ZipFile, writer: int):
    assert writer in ORIGINAL_WRITERS
    names = expected_files(writer)
    files = [n for n in archive.namelist() if n.lower().endswith('.jpg')]
    by_name = {Path(n).name: n for n in files}
    if len(files) != 640 or len(by_name) != 640 or set(by_name) != set(names):
        raise ValueError(f'Writer {writer}: missing, extra or duplicate JPEGs')
    imgs = []
    for name in names:
        with Image.open(BytesIO(archive.read(by_name[name]))) as im:
            if im.size != (28, 28):
                raise ValueError(f'Writer {writer}: unexpected size in {name}: {im.size}')
            imgs.append(np.asarray(im.convert('L'), dtype=np.uint8))
    images = np.stack(imgs)
    if writer in INVERT_IDS:
        images = 255 - images
    if writer in ROTATE_IDS:
        images = np.rot90(images, k=1, axes=(1, 2)).copy()
    labels = np.array([name[0] for name in names], dtype='<U1')
    return images, labels, np.array([LABELS.index(c) for c in labels], dtype=np.uint8)


def read_replacement_42(archive: ZipFile):
    """Validate new 42's NPZ and JPEGs; use NPZ X without pixel modifications.

    The submitted JPGs have tiny lossy-encoding differences (max 2 gray levels)
    from the submitted NPZ's X, so X is the authoritative source for 42.
    """
    npzs = [n for n in archive.namelist() if Path(n).name == 'ml26_42.npz']
    if len(npzs) != 1:
        raise ValueError('Replacement ZIP must contain exactly one ml26_42.npz')
    names = expected_files(42)
    jpg_entries = [n for n in archive.namelist() if n.lower().endswith('.jpg')]
    jpg_by_name = {Path(n).name:n for n in jpg_entries}
    if len(jpg_entries) != 640 or len(jpg_by_name) != 640 or set(jpg_by_name) != set(names):
        raise ValueError('Replacement ZIP: unexpected 42 JPEG sample names/counts')
    with np.load(BytesIO(archive.read(npzs[0])), allow_pickle=False) as data:
        for k in ('X','y','images','labels','ids','label_names','filenames'):
            if k not in data:
                raise ValueError(f'Replacement 42 NPZ lacks {k}')
        img = data['X'].copy()
        y = data['y'].copy()
        if img.shape != (640,28,28) or img.dtype != np.uint8:
            raise ValueError('Replacement 42 images must be uint8[640,28,28]')
        if y.shape != (640,) or y.dtype != np.uint8:
            raise ValueError('Replacement 42 y must be uint8[640]')
        if not (np.array_equal(img, data['images']) and np.array_equal(y, data['labels'])):
            raise ValueError('Replacement 42 redundant image/label arrays disagree')
        if (data['filenames'].astype(str).tolist() != names or
            data['label_names'].astype(str).tolist() != list(LABELS) or
            not np.array_equal(data['ids'],np.repeat(np.arange(1,41,dtype=np.uint8),16))):
            raise ValueError('Replacement 42 ordering/IDs do not match standard')
    expected_y = np.tile(np.arange(16,dtype=np.uint8),40)
    if not np.array_equal(y, expected_y):
        raise ValueError('Replacement 42 labels inconsistent with filenames')
    for index, name in enumerate(names):
        with Image.open(BytesIO(archive.read(jpg_by_name[name]))) as im:
            if im.size != (28,28) or im.mode != 'L':
                raise ValueError(f'Replacement 42 is not 28x28 grayscale: {name}')
            jpg = np.asarray(im, dtype=np.uint8)
            if np.max(np.abs(jpg.astype(np.int16)-img[index].astype(np.int16))) > 2:
                raise ValueError(f'Replacement 42 JPG vs NPZ pixel mismatch >2: {name}')
    labels = np.array([LABELS[i] for i in y],dtype='<U1')
    return img, labels, y


def write_writer(images, labels, ids, writer: int, output_dir: Path):
    if images.shape != (640,28,28) or images.dtype != np.uint8:
        raise AssertionError('Image size/dtype invalid')
    if len(ids) != 640 or Counter(labels) != {c:40 for c in LABELS}:
        raise AssertionError('Labels invalid')
    path = output_dir / f'ml26_{writer}.npz'
    np.savez_compressed(path, img=images, label=labels, label_id=ids)
    with np.load(path,allow_pickle=False) as check:
        if (check.files != ['img','label','label_id'] or
            not np.array_equal(check['img'],images) or
            not np.array_equal(check['label'],labels) or
            not np.array_equal(check['label_id'],ids)):
            raise AssertionError('NPZ round-trip failed')
    digest=sha256(path.read_bytes()).hexdigest()
    print(f'writer={writer} shape={images.shape} unique={len(np.unique(images))} '
          f'inverted={writer in INVERT_IDS} rotated_ccw90={writer in ROTATE_IDS} sha256={digest}')
    return path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument('--source',type=Path,help='ML-Project.zip with original writer 41/43/44 ZIPs')
    group.add_argument('--input-dir',type=Path,help='Directory with original ml26_41.zip, ml26_43.zip, ml26_44.zip')
    parser.add_argument('--writer42-replacement',type=Path,required=True,
                        help='New corrected 42 ZIP containing the authoritative X array; required to prevent old-data overwrite')
    parser.add_argument('--output-dir',type=Path,default=Path('02_data/Hand_made'))
    args=parser.parse_args()
    if not args.writer42_replacement.is_file():
        parser.error(f'Corrected writer42 ZIP not found: {args.writer42_replacement}')
    args.output_dir.mkdir(exist_ok=True,parents=True)
    # Read & validate ALL four inputs before writing anything.
    inputs = {}
    if args.source:
        with ZipFile(args.source) as bundle:
            for writer in ORIGINAL_WRITERS:
                matches=[n for n in bundle.namelist() if n.endswith(f'/ml26_{writer}.zip')]
                if len(matches)!=1:
                    raise ValueError(f'Expected one original ZIP for {writer}')
                with ZipFile(BytesIO(bundle.read(matches[0]))) as original:
                    inputs[writer]=read_original_writer(original,writer)
    else:
        for writer in ORIGINAL_WRITERS:
            with ZipFile(args.input_dir / f'ml26_{writer}.zip') as original:
                inputs[writer]=read_original_writer(original,writer)
    with ZipFile(args.writer42_replacement) as replacement:
        inputs[42]=read_replacement_42(replacement)
    for writer in WRITERS:
        write_writer(*inputs[writer],writer,args.output_dir)


if __name__ == '__main__':
    main()

## rebuild_ml04_v3

원본: [02_data/scripts/rebuild_ml04_v3.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/02_data/scripts/rebuild_ml04_v3.py)  
Git blob: `15aec4a103898860629814ab87e6b10c9572aff8`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/02_data)

In [ ]:
%%archive_source 02_data/scripts/rebuild_ml04_v3.py
#!/usr/bin/env python3
"""Rebuild and verify the reviewed v3 NPZ from the official 112,000 image baseline.

Requires Python 3.13, NumPy 2.3.5, SciPy 1.17.0.
Stops without modifying the original unless the resulting file exactly matches
the reviewed v3 SHA-256. This is a one-time migration, not a training pipeline.
"""
import hashlib
from collections import Counter
from pathlib import Path
import numpy as np
from scipy.ndimage import maximum_filter, minimum_filter, rotate

DATASET = Path("02_data/datasets/MNIST_EMNIST_official_split_112000.npz")
ORIGINAL_HASH = "b52ce7873abb21182627841fb22b5c87fd6f54ac5cb925242deab44499271ec1"
REVIEWED_HASH = "0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879"
SEED = 20261107

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def imghash(im):
    return hashlib.blake2b(np.ascontiguousarray(im).tobytes(), digest_size=16).digest()

def duplicates(train, test):
    a, b = Counter(map(imghash, train)), Counter(map(imghash, test))
    return sum(n*(n-1)//2 for n in a.values()), sum(n*b.get(h,0) for h,n in a.items())

def border_max(x):
    return int(max(np.max(x[0,:]), np.max(x[-1,:]), np.max(x[:,0]), np.max(x[:,-1])))

def transform(original, method, rng):
    a = original.astype(np.float32)
    if method == 1:
        strength = float(rng.choice([-1.,1.]))*float(rng.uniform(3.0,9.0))
        image = rotate(a,strength,reshape=False,order=1,mode="constant",cval=0.,prefilter=False)
    elif method == 2:
        strength = float(rng.uniform(0.22,0.45))
        image = (1-strength)*a + strength*maximum_filter(a,size=3,mode="constant",cval=0)
    elif method == 3:
        strength = float(rng.uniform(0.20,0.42))
        image = (1-strength)*a + strength*minimum_filter(a,size=3,mode="constant",cval=0)
    else: raise ValueError(method)
    return np.rint(np.clip(image,0,255)).astype(np.uint8),strength

def quality_ok(orig,new,method):
    before,after=int(orig.sum()),int(new.sum())
    if before==0 or after==0 or np.array_equal(orig,new):return False
    low,high={1:(0.90,1.08),2:(1.05,1.53),3:(0.57,0.97)}[method]
    if not low<=after/before<=high:return False
    if border_max(orig)<35 and border_max(new)>=100:return False
    return True

def main():
    actual=sha256(DATASET)
    if actual == REVIEWED_HASH:
        print("Already v3; no changes")
        return
    if actual != ORIGINAL_HASH:
        raise RuntimeError(f"Unknown NPZ version: {actual}")
    with np.load(DATASET,allow_pickle=False) as f:
        d={k:f[k].copy() for k in f.files}
    tr,te=d["X_train"],d["X_test"]
    ytr,yte=d["y_train"],d["y_test"]
    aug=d["train_is_augmented"]
    assert tr.shape==(96000,28,28) and te.shape==(16000,28,28)
    assert int(aug.sum())==5348 and duplicates(tr,te)[1]==1
    assert int(ytr[78001])==int(ytr[79394])==int(yte[13962])==13
    assert np.array_equal(tr[79394],te[13962])
    assert not aug[78001] and not aug[79394]

    # v2: swap the D original at train[78001] with test[13962].
    for tk,ek in [("X_train","X_test"),("train_source_index","test_source_index"),
                  ("train_source_split","test_source_split")]:
        tmp=d[tk][78001].copy()
        d[tk][78001]=d[ek][13962]
        d[ek][13962]=tmp

    source=d["train_source_index"]
    split=d["train_source_split"]
    lookup={(int(ytr[i]),int(split[i]),int(source[i])):int(i)
            for i in np.flatnonzero(~aug)}
    assert len(lookup)==int((~aug).sum())
    rng=np.random.default_rng(SEED)
    types=np.zeros(len(tr),dtype=np.uint8)
    strengths=np.zeros(len(tr),dtype=np.float32)
    aug_ids=np.flatnonzero(aug)
    for label in (11,13,14):
        ids=aug_ids[ytr[aug_ids]==label]
        counts=[round(len(ids)*0.40),round(len(ids)*0.30)]
        counts += [len(ids)-sum(counts)]
        recipe=np.repeat([1,2,3],counts)
        rng.shuffle(recipe)
        types[ids]=recipe
    assert np.all(types[aug_ids]!=0)

    seen={imghash(im) for im in tr[~aug]}
    seen.update(imghash(im) for im in te)
    for i in aug_ids:
        key=(int(ytr[i]),int(split[i]),int(source[i]))
        parent=tr[lookup[key]]
        requested=int(types[i])
        for attempt in range(100):
            candidate,strength=transform(parent,requested,rng)
            if quality_ok(parent,candidate,requested) and imghash(candidate) not in seen:
                break
        else:raise RuntimeError(f"Augmentation failed at {i}")
        tr[i]=candidate
        strengths[i]=strength
        seen.add(imghash(candidate))
    d["train_augmentation_type"]=types
    d["train_augmentation_strength"]=strengths
    d["augmentation_seed"]=np.array(SEED,dtype=np.int64)
    d["dataset_revision"]=np.array("v2: split_duplicate_swap+rotation+stroke")
    assert duplicates(tr,te)==(1,0)

    # v3: correct the remaining duplicate inside Training only.
    i,j=78001,79394
    assert np.array_equal(tr[i],tr[j]) and not aug[i]
    image=rotate(tr[j].astype(np.float32),5.0,reshape=False,order=1,mode="constant",cval=0.0)
    tr[i]=np.rint(np.clip(image,0,255)).astype(np.uint8)
    aug[i]=True
    d["train_source_split"][i]=d["train_source_split"][j]
    d["train_source_index"][i]=d["train_source_index"][j]
    types[i]=1
    strengths[i]=5.0
    d["dataset_revision"]=np.asarray("v3: single_D_train_dup_aug_replacement")
    assert duplicates(tr,te)==(0,0)
    assert int(aug.sum())==5349
    assert Counter(types[aug].tolist())==Counter({1:2141,2:1604,3:1604})
    for label,expected in [(11,2474),(13,1660),(14,1215)]:
        assert int(((ytr==label)&aug).sum())==expected
    tmp=DATASET.with_name(DATASET.stem+"_tmp.npz")
    np.savez_compressed(tmp,**d)
    checksum=sha256(tmp)
    if checksum!=REVIEWED_HASH:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"v3 hash mismatch: {checksum}")
    tmp.replace(DATASET)
    print(f"Updated {DATASET}: SHA256={sha256(DATASET)}, 96000/16000; augmented=5349; exact duplicates=0")

if __name__=="__main__":main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")